In [1]:
#!pip install UnityPy
#!pip uninstall -y UnityPy
#!pip install git+https://github.com/dv1x3r/UnityPy.git@test

In [2]:
import os
import json
import base64
import UnityPy
from collections import Counter

In [3]:
def cdnid_to_oid(cdnid):
    try:
        padded = cdnid + "=" * (-len(cdnid) % 4)
        decoded = base64.b64decode(padded)
        return int(decoded.decode('ascii'))
    except:
        print("unable to get oid from cdnid:", cdnid)
        return 0

In [4]:
export_path = "../../data_db/exported"
file_path = "../../data_db/assets/OTI4Mjg5ODYyNDUyNg" # gelato
#file_path = "../../data_db/assets/OTI4MzAxNzExMzYxNA" # karma
#file_path = "../../data_db/assets/Login.unity3d"
env = UnityPy.load(file_path)
vars(env)

{'files': {'../../data_db/assets/OTI4Mjg5ODYyNDUyNg': <BundleFile>},
 'cabs': {'cab-npcgelato': <SerializedFile>},
 'fs': <fsspec.implementations.local.LocalFileSystem at 0x1079c9400>,
 'local_files': [],
 'local_files_simple': [],
 'path': '../../data_db/assets',
 'file': <BundleFile>}

In [5]:
file = {
    "name": env.file.name,
    "type": "AssetBundle",
    "oid": cdnid_to_oid(env.file.name),
    "size": os.path.getsize(file_path), # size in bytes
}
file

{'name': 'OTI4Mjg5ODYyNDUyNg',
 'type': 'AssetBundle',
 'oid': 9282898624526,
 'size': 969251}

In [6]:
bundle = {
    "signature": env.file.signature, # bundle format
    "version": env.file.version, # format version
    "version_player": env.file.version_player,
    "version_engine": env.file.version_engine,
}
bundle

{'signature': 'UnityFS',
 'version': 6,
 'version_player': '5.x.x',
 'version_engine': '5.3.4p6'}

In [7]:
types = Counter()
for obj in env.objects:
    types[obj.type.name] += 1
    
counts = {
    "assets": len(env.assets),
    "objects": len(env.objects),
    "container": len(env.container),
    "types": dict(types),
}
counts

{'assets': 1,
 'objects': 126,
 'container': 25,
 'types': {'Transform': 45,
  'GameObject': 45,
  'AnimationClip': 16,
  'AudioSource': 1,
  'Material': 2,
  'Animation': 2,
  'MonoScript': 2,
  'Shader': 1,
  'MonoBehaviour': 2,
  'Texture2D': 2,
  'AssetBundle': 1,
  'SkinnedMeshRenderer': 1,
  'Rigidbody': 1,
  'Mesh': 1,
  'CapsuleCollider': 2,
  'MeshRenderer': 1,
  'MeshFilter': 1}}

In [8]:
env.file.files

{'CAB-NPCgelato': <SerializedFile>}

In [9]:
env.cabs

{'cab-npcgelato': <SerializedFile>}

In [10]:
# env.assets is a list of all SerializedFiles in the environment
assets = [{"name": asset.name, "target_platform": asset.target_platform.name} for asset in env.assets]
assets

[{'name': 'CAB-NPCgelato', 'target_platform': 'StandaloneWindows64'}]

In [11]:
containers = {v.m_PathID: k for k, v in env.container.items()}
containers

{4257807447455304617: 'assets/content/leveldesign/springtimezoneassets/characters/npcs/gelato/npcgelato.prefab',
 -1391753194372551919: 'assets/content/leveldesign/springtimezoneassets/characters/npcs/gelato/v.1_character/gelato.levelup.fbx',
 4088982439068189113: 'assets/content/leveldesign/springtimezoneassets/characters/npcs/gelato/v.1_character/skelgelatoanimation.fbx',
 2150093432316529690: 'assets/content/leveldesign/springtimezoneassets/characters/npcs/gelato/v.1_character/skelgelatoanimation.fbx',
 -4870540321401013829: 'assets/content/leveldesign/springtimezoneassets/characters/npcs/gelato/v.1_character/skelgelatoanimation.fbx',
 -2835988624527168518: 'assets/content/leveldesign/springtimezoneassets/characters/npcs/gelato/v.1_character/skelgelatoanimation.fbx',
 -6902679224432352475: 'assets/content/leveldesign/springtimezoneassets/characters/npcs/gelato/v.1_character/skelgelatoanimation.fbx',
 3895748274102892746: 'assets/content/leveldesign/springtimezoneassets/characters/np

In [12]:
roots = [] # transform [path_id]
children_dict = {} # transform {parent_id: [children_path_ids]}
transform_dict = {} # transform {path_id: parsed_as_object}
object_dict = {obj.path_id: obj for obj in env.objects}

for obj in env.objects:
    if obj.type.name == "Transform":
        tr = obj.parse_as_object()
        transform_dict[obj.path_id] = tr
        if not tr.m_Father:
            roots.append(obj.path_id)
        else:
            parent_id = tr.m_Father.path_id
            children_dict.setdefault(parent_id, []).append(obj.path_id)

def parse_audio_source(data):
    clip_id = data.m_audioClip.m_PathID
    clip = {"id": clip_id}
    if clip_obj := object_dict.get(clip_id):
        clip["name"] = clip_obj.peek_name()
    if clip_path := data.assets_file.container.path_dict.get(clip_id):
        clip["file"] = os.path.basename(clip_path)
    return clip

def parse_mono_behaviour(data):
    script_id = data.m_Script.m_PathID
    script = {"id": script_id}
    if script_obj := object_dict.get(script_id):
        script["name"] = script_obj.peek_name()
    if script_path := data.assets_file.container.path_dict.get(script_id):
        script["file"] = os.path.basename(script_path)
    if hasattr(data, "triggerEvent"):
        script["trigger_event"] = data.triggerEvent
    return script

def parse_mesh(data):
    mesh_id = data.m_Mesh.m_PathID
    mesh = {"id": mesh_id}
    if mesh_obj := object_dict.get(mesh_id):
        mesh["name"] = mesh_obj.peek_name()
    if mesh_path := data.assets_file.container.path_dict.get(mesh_id):
        mesh["file"] = os.path.basename(mesh_path)
    return mesh

def parse_material_textures(data):
    textures = []
    for tex_prop, tex_env in data.m_SavedProperties.m_TexEnvs:
        if tex_prop.name != "_MainTex":
            continue
        tex_id = tex_env.m_Texture.m_PathID
        texture = {"id": tex_id}
        if tex_obj := object_dict.get(tex_id):
            texture["name"] = tex_obj.peek_name()
        if tex_path := data.assets_file.container.path_dict.get(tex_id):
            texture["file"] = os.path.basename(tex_path)
        textures.append(texture)
    return textures
    
def parse_materials(data):
    materials = []
    for mat in data.m_Materials:
        mat_id = mat.m_PathID
        material = {"id": mat_id}
        if mat_obj := object_dict.get(mat_id):
            if mat_obj.type.name != "Material":
                continue
            mat_data = mat_obj.parse_as_object()
            material["name"] = mat_data.m_Name        
            if hasattr(mat_data, "m_SavedProperties"):
                material["textures"] = parse_material_textures(mat_data)
        if mat_path := data.assets_file.container.path_dict.get(mat_id):
            material["file"] = os.path.basename(mat_path)
        materials.append(material)
    return materials

def parse_animations(data):
    animations = []
    for anim in data.m_Animations:
        anim_id = anim.m_PathID
        animation = {"id": anim_id}
        if anim_obj := object_dict.get(anim_id):
            animation["name"] = anim_obj.peek_name()
        if anim_path := data.assets_file.container.path_dict.get(anim_id):
            animation["file"] = os.path.basename(anim_path)
        animations.append(animation)
    return animations

def get_components(game_object):
    components = []
    for _, comp in game_object.m_Component:
        comp_obj = object_dict.get(comp.m_PathID)
        component = {"type": comp_obj.type.name}
        if comp_obj.type.name == "AudioSource":
            data = comp_obj.parse_as_object()
            component["clip"] = parse_audio_source(data)
        elif comp_obj.type.name == "MonoBehaviour":
            data = comp_obj.parse_as_object()
            component["script"] = parse_mono_behaviour(data)
        elif comp_obj.type.name == "MeshFilter":
            data = comp_obj.parse_as_object()
            component["mesh"] = parse_mesh(data)
        elif comp_obj.type.name == "MeshRenderer":
            data = comp_obj.parse_as_object()
            if materials := parse_materials(data):
                component["materials"] = materials
        elif comp_obj.type.name == "SkinnedMeshRenderer":
            data = comp_obj.parse_as_object()
            component["mesh"] = parse_mesh(data)
            if materials := parse_materials(data):
                component["materials"] = materials
        elif comp_obj.type.name == "Animation":
            data = comp_obj.parse_as_object()
            if animations := parse_animations(data):
                component["animations"] = animations
        components.append(component)
    return components

def build_tree(tid):
    tr = transform_dict[tid]
    path_id = tr.m_GameObject.path_id
    game_object = tr.m_GameObject.read()
    node = {"id": path_id, "name": game_object.m_Name}
    if path := tr.assets_file.container.path_dict.get(path_id):
        node["file"] = os.path.basename(path)
    if components := get_components(game_object):
        node["components"] = components
    if children := [build_tree(child_id) for child_id in children_dict.get(tid, [])]:
        node["children"] = children
    return node

scene_hierarchy = [build_tree(root_id) for root_id in roots]
scene_hierarchy

[{'id': 4257807447455304617,
  'name': 'NPCgelato',
  'file': 'npcgelato.prefab',
  'components': [{'type': 'Transform'},
   {'type': 'Animation',
    'animations': [{'id': 4088982439068189113,
      'name': 'celebration',
      'file': 'skelgelatoanimation.fbx'},
     {'id': 2150093432316529690,
      'name': 'communication',
      'file': 'skelgelatoanimation.fbx'},
     {'id': -4870540321401013829,
      'name': 'frustration',
      'file': 'skelgelatoanimation.fbx'},
     {'id': -2835988624527168518,
      'name': 'greeting001',
      'file': 'skelgelatoanimation.fbx'},
     {'id': -6902679224432352475,
      'name': 'greeting002',
      'file': 'skelgelatoanimation.fbx'},
     {'id': 3895748274102892746,
      'name': 'greeting003',
      'file': 'skelgelatoanimation.fbx'},
     {'id': 205175828169347879,
      'name': 'greeting004',
      'file': 'skelgelatoanimation.fbx'},
     {'id': -6941572067104327796,
      'name': 'idle001',
      'file': 'skelgelatoanimation.fbx'},
     {

In [13]:
summary = {
    "file": file,
    "bundle": bundle,
    "counts": counts,
    "assets": assets,
    "containers": containers,
    "scene": scene_hierarchy,
}

In [14]:
os.makedirs(os.path.join(export_path, env.file.name, "audio"), exist_ok=True)
os.makedirs(os.path.join(export_path, env.file.name, "images"), exist_ok=True)
os.makedirs(os.path.join(export_path, env.file.name, "models"), exist_ok=True)

with open(os.path.join(export_path, env.file.name+".json"), "w") as f:
    json.dump(summary, f, indent=2)
    
exported = Counter()
other = Counter()

for obj in env.objects:    
    if obj.type.name == "AudioClip":
        clip = obj.parse_as_object()
        for sample_name, sample_data in clip.samples.items():
            # filename is .ogg on linux? todo: mp3
            # Login.unity3d is in .ogg
            file_name = f"{obj.path_id}.{sample_name}"
            file_path = os.path.join(export_path, env.file.name, "audio", file_name)
            with open(file_path, "wb") as f:
                f.write(sample_data)
            exported[obj.type.name] += 1
    
    elif obj.type.name == "Mesh":
        data = obj.parse_as_object()
        file_name = f"{obj.path_id}.{data.m_Name}.obj"
        file_path = os.path.join(export_path, env.file.name, "models", file_name)
        if wavefront_data := data.export():
            with open(file_path, "wt", newline="") as f:
                f.write(wavefront_data)
            exported[obj.type.name] += 1
        else:
            print("invalid mesh:", file_name)
    
    elif obj.type.name in ["Texture2D", "Sprite"]:
        data = obj.parse_as_object()
        file_name = f"{obj.path_id}.{data.m_Name}.png"
        file_path = os.path.join(export_path, env.file.name, "images", file_name)
        data.image.save(file_path)
        exported[obj.type.name] += 1

    else:
        other[obj.type.name] += 1
        
print('exported:', dict(exported))
print('other:', dict(other))

exported: {'Texture2D': 2, 'Mesh': 1}
other: {'Transform': 45, 'GameObject': 45, 'AnimationClip': 16, 'AudioSource': 1, 'Material': 2, 'Animation': 2, 'MonoScript': 2, 'Shader': 1, 'MonoBehaviour': 2, 'AssetBundle': 1, 'SkinnedMeshRenderer': 1, 'Rigidbody': 1, 'CapsuleCollider': 2, 'MeshRenderer': 1, 'MeshFilter': 1}
